## Exercise 2

In [ ]:
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import ExtraTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt
import numpy as np


# from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
# from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

### 2.1 Import Outage Data

In [ ]:
%%time

csvPath = "C:/Users/TJones/OneDrive/Documents/IEEE/Presentations/2026 T&D/2026 TD Tutorial/Tyler's presentation material/outage_data.csv"
df = pd.read_csv(csvPath)

# print(df)
print(df.columns.tolist())

## new line before the final %%time
print("")

### 2.2 Calculate IEEE TMED

1. sum CI and SAIDI by day
2. filter where sum CI > 0
3. add the ln(SAIDI) column
4. calculate Alpha and Beta
5. calculate TMED


In [ ]:
daily_data = (
  df
    .groupby("day", as_index=False)
    .agg(
      ci=("ci", "sum"),
      saidi=("saidi", "sum")
    )
)

alpha = np.log(daily_data.loc[daily_data["saidi"] > 0, "saidi"]).mean()
beta = np.log(daily_data.loc[daily_data["saidi"] > 0, "saidi"]).std()

tmed = np.exp(alpha + 2.5 * beta)

print(f"Alpha: {alpha:.5f}")
print(f"Beta :  {beta:.5f}")
print(f"TMED :  {tmed:.2f}")

### 2.3 Identify a Major Event

In [ ]:
# using the "daily_data" data frame from 1.2
major_event_days = daily_data[daily_data["saidi"] >= tmed].copy()
# append tmed so it can be seen
major_event_days["tmed"] = tmed

print(major_event_days)

### 2.4 MED Intervals

In [ ]:
df["outage_start"] = pd.to_datetime(df["outage_start"])
df["outage_end"] = pd.to_datetime(df["outage_end"])
df["day"] = df["outage_start"].dt.floor("D")

major_event_days["day"] = pd.to_datetime(major_event_days["day"]).dt.floor("D")

for event_day in sorted(major_event_days["day"].dropna().unique()):
  event_day = pd.Timestamp(event_day)

  start = event_day - pd.Timedelta(days=1)
  end = event_day + pd.Timedelta(days=1)

  event_window = df[
    (df["outage_start"] >= start) &
    (df["outage_start"] < end)
  ].copy()

  if event_window.empty:
    continue

  event_window = event_window.sort_values("outage_start")
  event_window["cumulative_saidi"] = event_window["saidi"].cumsum()

  starts = event_window[["outage_start"]].rename(columns={"outage_start": "timestamp"})
  starts["delta"] = 1

  ends = event_window[["outage_end"]].rename(columns={"outage_end": "timestamp"})
  ends["delta"] = -1

  outage_count = (
    pd.concat([starts, ends], ignore_index=True)
    .dropna(subset=["timestamp"])
    .sort_values("timestamp")
  )

  outage_count["active_outage_count"] = outage_count["delta"].cumsum()

  fig, ax1 = plt.subplots(figsize=(12, 5))

  ax1.plot(
    event_window["outage_start"],
    event_window["cumulative_saidi"],
    marker="o",
    linewidth=1,
    label="Cumulative SAIDI"
  )

  ax1.axhline(
    y=tmed,
    linestyle="--",
    linewidth=2,
    label=f"TMED = {tmed:.2f}"
  )

  ax1.set_title(f"Cumulative SAIDI Around Major Event Day: {event_day.date()}")
  ax1.set_xlabel("Time")
  ax1.set_ylabel("Cumulative SAIDI")
  ax1.grid(True, alpha=0.3)

  ax2 = ax1.twinx()

  ax2.step(
    outage_count["timestamp"],
    outage_count["active_outage_count"],
    where="post",
    linewidth=1,
    label="Active Outage Count"
  )

  ax2.set_ylabel("Active Outage Count")

  lines1, labels1 = ax1.get_legend_handles_labels()
  lines2, labels2 = ax2.get_legend_handles_labels()
  ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

  plt.tight_layout()
  plt.show()

### 2.5 Rolling 365 Charts

In [ ]:
df["outage_start"] = pd.to_datetime(df["outage_start"])
df["day"] = df["outage_start"].dt.floor("D")

major_event_days["day"] = pd.to_datetime(major_event_days["day"]).dt.floor("D")

df = df.sort_values("outage_start").copy()

df["rolling_365_saidi"] = (
  df["saidi"]
    .rolling(window=365, min_periods=365)
    .sum()
)

df_excl_med = (
  df[~df["day"].isin(major_event_days["day"])]
    .sort_values("outage_start")
    .copy()
)

df_excl_med["rolling_365_saidi_excl_med"] = (
  df_excl_med["saidi"]
    .rolling(window=365, min_periods=365)
    .sum()
)

plt.figure(figsize=(12, 5))

plt.plot(
  df["outage_start"],
  df["rolling_365_saidi"],
  linewidth=1,
  label="Rolling 365 SAIDI"
)

plt.plot(
  df_excl_med["outage_start"],
  df_excl_med["rolling_365_saidi_excl_med"],
  linewidth=1,
  label="Rolling 365 SAIDI Excluding MEDs"
)

plt.title("Rolling 365-Row SAIDI With and Without Major Event Days")
plt.xlabel("Outage Start")
plt.ylabel("Rolling 365 SAIDI")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.6 Year-over-Year

In [ ]:
annual = (
  df.groupby("year", as_index=False)
    .agg(
      saidi=("saidi", "sum"),
      saifi=("saifi", "sum")
    )
)

annual["caidi"] = annual["saidi"] / annual["saifi"]

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(
  annual["year"],
  annual["saidi"],
  marker="o",
  linewidth=2,
  label="SAIDI"
)

ax1.plot(
  annual["year"],
  annual["caidi"],
  marker="o",
  linewidth=2,
  label="CAIDI"
)

ax1.set_xlabel("Year")
ax1.set_ylabel("SAIDI / CAIDI")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()

ax2.plot(
  annual["year"],
  annual["saifi"],
  marker="x",
  linewidth=2,
  label="SAIFI"
)

ax2.set_ylabel("SAIFI")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Annual SAIDI, SAIFI, and CAIDI")
plt.tight_layout()
plt.show()